# FYS4480/FYS9480 -- Week 44: many-body perturbation theory

Companion notebook to the week-44 slides (lecture notes chapter 9, sections
9.1-9.6).  Plain NumPy, self-contained; the same calculations as
`BookPrograms/chapter09/mbpt.py`, written out from scratch so that every
number on the slides can be checked.

1. The Rayleigh-Schrödinger recursion for an arbitrary matrix, and the
   explicit resolvent expressions through fourth order.
2. The two-state model with $H_I=g\sum_{pq}a_p^\dagger a_q$: orders read off
   the exact eigenvalue, recovered by the recursion; $E^{(3)}=0$ by a
   cancellation.
3. Brillouin-Wigner: the two-vertex equation, the $Q$-space resummation, and
   the secular equation.
4. The purely off-diagonal benchmark: the fourth order is the folded term;
   the radius of convergence $|2\lambda/\Delta|=1$.
5. The pairing model with four levels and two pairs: $E^{(2)}=-\tfrac{7}{24}g^2$,
   the singles sector empty, the counting factor.
6. Exercise 2 of this week, checked numerically: two particles in a single
   $j$-shell, $E_0=-(j+\tfrac12)G$ from Brillouin-Wigner and from
   diagonalisation; why Rayleigh-Schrödinger cannot be used.


In [1]:
import numpy as np
from itertools import combinations
np.set_printoptions(precision=8, suppress=True, linewidth=120)

## 1. The recursion, and the explicit expressions

Everything is done in a basis in which $H_0$ is diagonal, $H=H_0+V$, with
the reference as state 0 and the reduced resolvent $R=Q/(\varepsilon_0-H_0)$
a diagonal matrix that vanishes on the reference.  The recursion

$$|\Psi^{(n)}\rangle=R\Big[V|\Psi^{(n-1)}\rangle-\sum_{k=1}^{n-1}E^{(k)}|\Psi^{(n-k)}\rangle\Big],
\qquad E^{(n+1)}=\langle\Phi_0|V|\Psi^{(n)}\rangle$$

generates every order; the explicit expressions through fourth order are
written out separately so that the two can be compared.

In [2]:
def rayleigh_schrodinger(H0diag, V, order, ref=0):
    """RS energies E^(1..order) and wave-function corrections psi^(1..order-1)."""
    H0diag = np.asarray(H0diag, float); V = np.asarray(V, float)
    dim = H0diag.size
    R = np.zeros(dim); mask = np.arange(dim) != ref
    R[mask] = 1.0 / (H0diag[ref] - H0diag[mask])
    psi0 = np.zeros(dim); psi0[ref] = 1.0
    E = [float(psi0 @ V @ psi0)]
    psis = []
    for n in range(1, order):
        prev = psis[-1] if psis else psi0
        rhs = V @ prev
        for k in range(1, n):
            rhs = rhs - E[k - 1] * psis[n - k - 1]
        psis.append(R * rhs)
        E.append(float(psi0 @ V @ psis[-1]))
    return E, psis, R

def explicit_orders(H0diag, V, ref=0):
    """E^(1..4) from the resolvent expressions of the slides."""
    H0diag = np.asarray(H0diag, float); V = np.asarray(V, float)
    dim = H0diag.size
    R = np.zeros(dim); mask = np.arange(dim) != ref
    R[mask] = 1.0 / (H0diag[ref] - H0diag[mask])
    v0 = V[ref]
    av = lambda x: float(v0 @ x)             # <Phi_0| V x
    E1 = float(V[ref, ref])
    E2 = av(R * v0)
    E3 = av(R * (V @ (R * v0))) - E1 * av(R * R * v0)
    rv = R * v0; rvrv = R * (V @ rv); rvrvrv = R * (V @ rvrv)
    E4 = (av(rvrvrv)
          - E1 * (av(R * R * (V @ rv)) + av(R * (V @ (R * R * v0))))
          + E1**2 * av(R**3 * v0)
          - E2 * av(R * R * v0))
    return [E1, E2, E3, E4]

# a random check: symmetric V, non-degenerate H0
rng = np.random.default_rng(1)
H0 = np.sort(rng.uniform(0, 10, 7)); V = rng.normal(size=(7, 7)) * 0.3; V = (V + V.T) / 2
E_rec, _, _ = rayleigh_schrodinger(H0, V, 4)
E_exp = explicit_orders(H0, V)
print("recursion :", np.array(E_rec))
print("explicit  :", np.array(E_exp))
print("max difference:", np.abs(np.array(E_rec) - np.array(E_exp)).max())

recursion : [ 0.17433543 -0.00867248 -0.00044361 -0.00016577]
explicit  : [ 0.17433543 -0.00867248 -0.00044361 -0.00016577]
max difference: 0.0


## 2. The two-state model with a diagonal and an off-diagonal coupling

$H_0=\mathrm{diag}(\varepsilon_1,\varepsilon_2)$, $H_I=g\sum_{pq}a_p^\dagger a_q$,
so every matrix element of $H_I$ equals $g$.  The exact lower eigenvalue
expanded in $g$ gives $\varepsilon_1+g-g^2/\Delta+0+g^4/\Delta^3+\dots$; the
recursion must reproduce exactly these coefficients, and the third order must
vanish because the principal term $g^3/D^2$ cancels against the normalisation
subtraction $E^{(1)}\langle VR^2V\rangle=g^3/D^2$.

In [3]:
eps1, eps2, g = 0.0, 1.0, 0.2
Delta = eps2 - eps1
H0 = np.array([eps1, eps2]); V = g * np.ones((2, 2))
E_exact = (eps1 + eps2) / 2 + g - np.sqrt(Delta**2 / 4 + g**2)
E, psis, R = rayleigh_schrodinger(H0, V, 6)
print("exact E_-            :", E_exact)
print("expected orders      :", [g, -g**2 / Delta, 0.0, g**4 / Delta**3])
print("recursion E^(1..6)   :", np.array(E))
print("partial sums - exact :", np.cumsum(E) + eps1 - E_exact)
D = eps1 - eps2
print("third order: principal g^3/D^2 =", g**3 / D**2, " subtraction E1 <VR^2V> =", g * g**2 / D**2)

exact E_-            : 0.16148351928654947
expected orders      : [0.2, -0.04000000000000001, 0.0, 0.0016000000000000003]
recursion E^(1..6)   : [ 0.2      -0.04      0.        0.0016    0.       -0.000128]
partial sums - exact : [ 0.03851648 -0.00148352 -0.00148352  0.00011648  0.00011648 -0.00001152]
third order: principal g^3/D^2 = 0.008000000000000002  subtraction E1 <VR^2V> = 0.008000000000000002


## 3. Brillouin-Wigner: two vertices, the resummation, the secular equation

With one state in $Q$, the resummed equation
$E_0-\varepsilon_1=g+g^2/(E_0-\varepsilon_2-g)$ is the secular equation
itself.  The two-vertex equation *without* resummation,
$E_0-\varepsilon_1=g+g^2/(E_0-\varepsilon_2)$, is not, because it omits the
diagonal interaction $V_{22}=g$ inside $Q$; both are solved by iteration.

In [4]:
def bw_iterate(f, start, tol=1e-14, maxit=500):
    E = start
    for it in range(maxit):
        new = f(E)
        if abs(new - E) < tol:
            return new, it + 1
        E = new
    return E, maxit

resummed, it1 = bw_iterate(lambda E: eps1 + g + g**2 / (E - eps2 - g), eps1)
twovertex, it2 = bw_iterate(lambda E: eps1 + g + g**2 / (E - eps2), eps1)
print(f"BW resummed      : {resummed:.12f}  ({it1} iterations)   exact {E_exact:.12f}")
print(f"BW two vertices  : {twovertex:.12f}  ({it2} iterations)   not exact: V_22 = g is missing inside Q")
# the resummed equation multiplied out is det(H - E) = 0
Hm = np.array([[eps1 + g, g], [g, eps2 + g]])
print("det(H - E_resummed) =", np.linalg.det(Hm - resummed * np.eye(2)))

BW resummed      : 0.161483519287  (11 iterations)   exact 0.161483519287
BW two vertices  : 0.152786404500  (12 iterations)   not exact: V_22 = g is missing inside Q
det(H - E_resummed) = -3.3306690738754485e-17


## 4. The purely off-diagonal benchmark through fourth order

$V_{11}=V_{22}=0$, $V_{12}=\lambda$.  Now $QVQ=0$, so the two-vertex
Brillouin-Wigner equation is already exact, the Rayleigh-Schrödinger third
order vanishes, and the whole fourth order is the folded term
$-E^{(2)}\langle VR^2V\rangle=-\lambda^4/D^3$.  The exact eigenvalue
$\varepsilon_1+\tfrac{\Delta}{2}[1-\sqrt{1+4\lambda^2/\Delta^2}]$ has branch
points at $\lambda=\pm i\Delta/2$, so the series converges only for
$|2\lambda/\Delta|<1$.

In [5]:
lam = 0.2
V = np.array([[0.0, lam], [lam, 0.0]])
E, psis, R = rayleigh_schrodinger(H0, V, 8)
exact = eps1 + Delta / 2 * (1 - np.sqrt(1 + 4 * lam**2 / Delta**2))
print("E^(1..8):", np.array(E))
print("expected: E2 =", lam**2 / D, " E4 =", -lam**4 / D**3)
# the four pieces of the fourth order
v0 = V[0]; av = lambda x: float(v0 @ x)
rv = R * v0; rvrv = R * (V @ rv); rvrvrv = R * (V @ rvrv)
E1, E2 = float(V[0, 0]), av(R * v0)
print("principal <VRVRVRV> =", av(rvrvrv), "  mixed =", -E1 * (av(R*R*(V@rv)) + av(R*(V@(R*R*v0)))),
      "  squared =", E1**2 * av(R**3 * v0), "  folded =", -E2 * av(R * R * v0))
bw, it = bw_iterate(lambda E: eps1 + lam**2 / (E - eps2), eps1)
print(f"BW two vertices: {bw:.12f} ({it} iterations)   exact {exact:.12f}")

E^(1..8): [ 0.        -0.04       0.         0.0016     0.        -0.000128   0.         0.0000128]
expected: E2 = -0.04000000000000001  E4 = 0.0016000000000000003
principal <VRVRVRV> = 0.0   mixed = -0.0   squared = -0.0   folded = 0.0016000000000000007
BW two vertices: -0.038516480713 (10 iterations)   exact -0.038516480713


In [6]:
print(f"{'lam/Delta':>10s} {'exact':>12s} {'order 2':>12s} {'order 4':>12s} {'order 8':>12s} {'order 16':>12s} {'order 30':>12s}")
for ratio in (0.1, 0.3, 0.45, 0.5, 0.55, 0.7):
    V = np.array([[0.0, ratio], [ratio, 0.0]])
    E, _, _ = rayleigh_schrodinger(H0, V, 30)
    sums = np.cumsum(E) + eps1
    exact = eps1 + Delta / 2 * (1 - np.sqrt(1 + 4 * ratio**2))
    print(f"{ratio:10.2f} {exact:12.6f} {sums[1]:12.6f} {sums[3]:12.6f} {sums[7]:12.6f} {sums[15]:12.6f} {sums[29]:12.6f}")
print()
print("|E^(n)| for even n, lam/Delta = 0.55: the terms decrease until n = 16 and grow afterwards")
V = np.array([[0.0, 0.55], [0.55, 0.0]]); E, _, _ = rayleigh_schrodinger(H0, V, 30)
print(np.abs(np.array(E))[1::2])

 lam/Delta        exact      order 2      order 4      order 8     order 16     order 30
      0.10    -0.009902    -0.010000    -0.009900    -0.009902    -0.009902    -0.009902
      0.30    -0.083095    -0.090000    -0.081900    -0.083030    -0.083095    -0.083095
      0.45    -0.172681    -0.202500    -0.161494    -0.169694    -0.172195    -0.172726
      0.50    -0.207107    -0.250000    -0.187500    -0.199219    -0.204147    -0.208289
      0.55    -0.243303    -0.302500    -0.210994    -0.224488    -0.228285    -0.265999
      0.70    -0.360233    -0.490000    -0.249900    -0.196958     0.519349   -38.869223

|E^(n)| for even n, lam/Delta = 0.55: the terms decrease until n = 16 and grow afterwards
[0.3025     0.09150625 0.05536128 0.04186697 0.03546132 0.03218115 0.03059508 0.03007879 0.03032944 0.03119383
 0.03259756 0.03451266 0.03694182 0.03991036 0.04346238]


## 5. The pairing model: four levels, two pairs

Levels $p=1,\dots,4$ with $\varepsilon_p=p-1$, doubly degenerate; two pairs;
the interaction moves a pair between levels with matrix element $-g/2$ and
is $-g/2$ on the diagonal of every pair configuration.  The seniority-zero
space has six determinants.  The singles sector of $E^{(2)}$ is empty
($U_{ai}=0$: the interaction cannot connect the reference to a one-particle-
one-hole state), and the doubles sector gives $E^{(2)}=-\tfrac{7}{24}g^2$ --
the first iterate of the projected FCI equations of week 38.

In [7]:
def pairing_matrix(g, levels=4, pairs=2):
    configs = list(combinations(range(levels), pairs))
    dim = len(configs)
    H0 = np.array([2.0 * sum(p for p in c) for c in configs])      # eps_p = p - 1, two particles per pair
    V = np.zeros((dim, dim))
    for a, ca in enumerate(configs):
        for b, cb in enumerate(configs):
            if a == b:
                V[a, a] = -g / 2 * pairs                           # -g/2 for each occupied pair
            elif len(set(ca) ^ set(cb)) == 2:                      # one pair moved
                V[a, b] = -g / 2
    return H0, V, configs

for g in (0.25, 0.5, 1.0):
    H0p, Vp, configs = pairing_matrix(g)
    E, _, _ = rayleigh_schrodinger(H0p, Vp, 4)
    exact = np.linalg.eigvalsh(np.diag(H0p) + Vp)[0]
    print(f"g = {g:4.2f}: E1 = {E[0]:+.6f} (= -g)  E2 = {E[1]:+.8f} (-7g^2/24 = {-7*g*g/24:+.8f})  "
          f"E3 = {E[2]:+.8f}  E4 = {E[3]:+.8f}   W0+sum = {2+sum(E):.8f}   exact {exact:.8f}")
print()
print("reference", configs[0], "   the four double excitations and their denominators:")
for b, cb in enumerate(configs[1:], 1):
    if len(set(configs[0]) ^ set(cb)) == 2:
        print("  ", cb, "  D =", H0p[0] - H0p[b], "  contribution g^2/4 / D =", (0.5**2) / (H0p[0] - H0p[b]), "(g = 0.5)")

g = 0.25: E1 = -0.250000 (= -g)  E2 = -0.01822917 (-7g^2/24 = -0.01822917)  E3 = -0.00130208  E4 = -0.00002176   W0+sum = 1.73044699   exact 1.73045985
g = 0.50: E1 = -0.500000 (= -g)  E2 = -0.07291667 (-7g^2/24 = -0.07291667)  E3 = -0.01041667  E4 = -0.00034813   W0+sum = 1.41631854   exact 1.41677428
g = 1.00: E1 = -1.000000 (= -g)  E2 = -0.29166667 (-7g^2/24 = -0.29166667)  E3 = -0.08333333  E4 = -0.00557002   W0+sum = 0.61942998   exact 0.63554847

reference (0, 1)    the four double excitations and their denominators:
   (0, 2)   D = -2.0   contribution g^2/4 / D = -0.125 (g = 0.5)
   (0, 3)   D = -4.0   contribution g^2/4 / D = -0.0625 (g = 0.5)
   (1, 2)   D = -4.0   contribution g^2/4 / D = -0.0625 (g = 0.5)
   (1, 3)   D = -6.0   contribution g^2/4 / D = -0.041666666666666664 (g = 0.5)


## 6. Exercise 2 of week 44, checked numerically

Two particles in a single $j$-shell, pair states $|m,-m\rangle$ for
$m=\tfrac12,\dots,j$, i.e.\ $\Omega=j+\tfrac12$ states, all degenerate in
$H_0$ (set $\varepsilon=0$), interaction $-G$ between any two pair states.
Diagonalisation gives $E_0=-\Omega G$ (the all-ones matrix has one eigenvalue
$\Omega$ and the rest zero).  Brillouin-Wigner with the full $Q$-space
resummation reproduces it: with $v_0=(-G,\dots,-G)$ an eigenvector of $QVQ$,
$E=-G+(\Omega-1)G^2/(E+(\Omega-1)G)$, whose solutions are $E=0$ and
$E=-\Omega G$.  Rayleigh-Schrödinger has $\varepsilon_0=\varepsilon_m$ for
every $m$: every denominator vanishes, and the non-degenerate formalism of
this week does not apply.

In [8]:
for j2 in (5, 7, 9):            # 2j = 5, 7, 9 -> j = 5/2, 7/2, 9/2
    Omega = (j2 + 1) // 2
    G = 1.0
    H = -G * np.ones((Omega, Omega))
    exact = np.linalg.eigvalsh(H)[0]
    # BW resummed on the degenerate shell: E = V00 + v0 (E - QVQ)^{-1} v0
    # v0 = (-G, ..., -G) is an eigenvector of Q V Q with eigenvalue -(Omega-1) G, so the
    # Q-space inverse acting on v0 is a number and the resummed BW equation is scalar:
    f = lambda E: -G + (Omega - 1) * G**2 / (E + (Omega - 1) * G)
    F = lambda E: E - f(E)                       # the BW equation written as F(E) = 0
    def bisect(a, b, tol=1e-13):
        fa = F(a)
        for _ in range(200):
            m = 0.5 * (a + b); fm = F(m)
            if fa * fm <= 0: b = m
            else: a, fa = m, fm
            if b - a < tol: break
        return 0.5 * (a + b)
    root_low = bisect(-Omega * G - 1.0, -Omega * G + 0.4)
    root_zero = bisect(-0.31, 0.3)
    E_iter, it = bw_iterate(f, -Omega * G - 0.37)
    print(f"j = {j2}/2, Omega = {Omega}: exact {exact:+.10f}   -(j+1/2)G = {-Omega*G:+.1f}   "
          f"BW roots {root_low:+.10f} and {root_zero:+.10f};  plain iteration from {-Omega*G-0.37} -> {E_iter:+.6f} ({it} it.)")
print()
print("The BW equation E = -G + (Omega-1) G^2 / (E + (Omega-1) G) multiplies out to E (E + Omega G) = 0:")
print("both roots are eigenvalues of H (0 is the (Omega-1)-fold degenerate excited level), and the plain")
print("fixed-point iteration is attracted to the wrong one -- the physical root is a repelling fixed point")
print("with |f'(E)| = Omega - 1.  Solve the BW equation as an equation, not by naive iteration.")
print()
print("Rayleigh-Schrodinger: eps_0 - eps_m = 0 for every m -- the reduced resolvent does not exist.")

j = 5/2, Omega = 3: exact -3.0000000000   -(j+1/2)G = -3.0   BW roots -3.0000000000 and -0.0000000000;  plain iteration from -3.37 -> -0.000000 (53 it.)
j = 7/2, Omega = 4: exact -4.0000000000   -(j+1/2)G = -4.0   BW roots -4.0000000000 and -0.0000000000;  plain iteration from -4.37 -> -0.000000 (35 it.)
j = 9/2, Omega = 5: exact -5.0000000000   -(j+1/2)G = -5.0   BW roots -5.0000000000 and -0.0000000000;  plain iteration from -5.37 -> +0.000000 (28 it.)

The BW equation E = -G + (Omega-1) G^2 / (E + (Omega-1) G) multiplies out to E (E + Omega G) = 0:
both roots are eigenvalues of H (0 is the (Omega-1)-fold degenerate excited level), and the plain
fixed-point iteration is attracted to the wrong one -- the physical root is a repelling fixed point
with |f'(E)| = Omega - 1.  Solve the BW equation as an equation, not by naive iteration.

Rayleigh-Schrodinger: eps_0 - eps_m = 0 for every m -- the reduced resolvent does not exist.


The lesson of the last cell is the one the week-44 slides state up front:
the whole construction assumes a *non-degenerate* reference, and the
degenerate shell is exactly the case where a model space of more than one
determinant -- and a different formalism -- is needed.